# Household Ecological-Economy — GSSK Notebook

This notebook runs the household multi-carrier model using the GSSK CLI and analyses:
1. Multi-carrier time series (money, energy, material, information)
2. Steady-state verification against the analytical solution
3. Cross-carrier sensitivity: how doubling `grocery_k` affects pantry steady-state

> **Requirements:** `make all` must be run first to build `./bin/gssk`.  
> Python packages: `pandas`, `matplotlib`, `numpy`.
> When the GSSK Python binding (`gsk-py`) ships in Phase 6.2, replace the subprocess calls with `import gssk`.

In [ ]:
import subprocess
import json
import tempfile
import os
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})

# Locate the repository root (notebook lives in examples/)
REPO = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'examples' else Path(os.getcwd())
GSSK_BIN  = REPO / 'bin' / 'gssk'
MODEL_JSON = REPO / 'examples' / 'household_model.json'

assert GSSK_BIN.exists(), f'Build the kernel first: make all (looking for {GSSK_BIN})'
assert MODEL_JSON.exists(), f'Model not found: {MODEL_JSON}'

print(f'GSSK binary : {GSSK_BIN}')
print(f'Model       : {MODEL_JSON}')

## Helper: run model and return a DataFrame

In [ ]:
def run_model(model_dict: dict) -> pd.DataFrame:
    """Serialise model_dict to JSON, run gssk, return time-series DataFrame."""
    with tempfile.NamedTemporaryFile(suffix='.json', mode='w', delete=False) as f:
        json.dump(model_dict, f, indent=2)
        model_path = f.name
    with tempfile.NamedTemporaryFile(suffix='.csv', delete=False) as f:
        csv_path = f.name
    try:
        result = subprocess.run(
            [str(GSSK_BIN), model_path, csv_path],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            raise RuntimeError(result.stderr or result.stdout)
        return pd.read_csv(csv_path)
    finally:
        os.unlink(model_path)
        os.unlink(csv_path)

# Load baseline model
with open(MODEL_JSON) as f:
    baseline = json.load(f)

df = run_model(baseline)
print(f'Rows: {len(df)}   Columns: {len(df.columns)}')
df.head(3)

## 1. Multi-Carrier Time Series

Four sub-plots — one per carrier — showing all storage nodes over 24 months.

In [ ]:
CARRIERS = {
    'Money (AUD)':    ['bank_account', 'super_fund', 'credit_card_debt'],
    'Energy (kWh)':   ['solar_battery', 'grid_credit', 'vehicle_fuel', 'body_energy'],
    'Material (kg)':  ['pantry', 'fridge', 'wardrobe', 'appliances_stock', 'waste_bin'],
    'Information':    ['household_attention', 'pending_decisions'],
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
fig.suptitle('Household Ecological-Economy — 24-Month Simulation', fontsize=13, fontweight='bold')

t = df['time']
for ax, (title, nodes) in zip(axes.flat, CARRIERS.items()):
    for node in nodes:
        ax.plot(t, df[node], label=node.replace('_', ' '))
    ax.set_title(title)
    ax.set_xlabel('Time (months)')
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(REPO / 'docs' / 'examples' / 'household' / 'timeseries.png', bbox_inches='tight')
plt.show()
print('Saved timeseries.png')

## 2. Steady-State Verification

**Analytical bank_account steady state:**

```
dQ_bank/dt = k_salary − k_mortgage − k_tax − k_spend × Q_bank
           = 5000 − 1800 − 600 − 0.1 × Q_bank = 0
→ Q* = 2600 / 0.1 = 26 000 AUD
```

The model runs for 24 months with τ = 10 months, so it has reached about
`1 − e^(−24/10) ≈ 91 %` of the way to steady state.

In [ ]:
# Analytical steady state
k_salary   = next(e['params']['k'] for e in baseline['edges'] if e['id'] == 'salary_in')
k_mortgage = next(e['params']['k'] for e in baseline['edges'] if e['id'] == 'mortgage_out')
k_tax      = next(e['params']['k'] for e in baseline['edges'] if e['id'] == 'tax_out')
k_spend    = next(e['params']['k'] for e in baseline['edges'] if e['id'] == 'grocery_payment')
Q0_bank    = next(n['value']       for n in baseline['nodes']  if n['id'] == 'bank_account')

Q_star = (k_salary - k_mortgage - k_tax) / k_spend
tau    = 1.0 / k_spend
t_arr  = df['time'].values
Q_analytical = Q_star + (Q0_bank - Q_star) * np.exp(-t_arr / tau)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_arr, df['bank_account'], label='GSSK simulation', lw=2)
ax.plot(t_arr, Q_analytical, '--', label=f'Analytical Q* = {Q_star:,.0f} AUD, τ = {tau:.0f} mo', lw=1.5)
ax.axhline(Q_star, color='grey', lw=0.8, ls=':', label='Steady state')
ax.set_xlabel('Time (months)')
ax.set_ylabel('Bank account (AUD)')
ax.set_title('Steady-State Verification: bank_account')
ax.legend()
ax.grid(alpha=0.3)

final_sim    = df['bank_account'].iloc[-1]
final_theory = Q_analytical[-1]
print(f'Simulation at t=24 : {final_sim:,.2f} AUD')
print(f'Analytical at t=24 : {final_theory:,.2f} AUD')
print(f'Relative error     : {abs(final_sim - final_theory) / final_theory * 100:.4f}%')

plt.tight_layout()
plt.savefig(REPO / 'docs' / 'examples' / 'household' / 'steady_state.png', bbox_inches='tight')
plt.show()

## 3. Cross-Carrier Sensitivity: grocery_k → pantry

The grocery_receive edge links money (bank_account) to material (pantry):

```
F_grocery = k_grocery × Q_market × Q_bank
```

At pantry steady state:

```
Q_pantry* ≈ (k_grocery × Q_bank*) / (k_spoil + k_ptof)  =  k_grocery × 26000 / 0.23
```

Prediction: doubling `k_grocery` should double `Q_pantry*`.

In [ ]:
def set_edge_k(model: dict, edge_id: str, new_k: float) -> dict:
    m = copy.deepcopy(model)
    for e in m['edges']:
        if e['id'] == edge_id:
            e['params']['k'] = new_k
    return m

# Sweep k_grocery
k_grocery_baseline = next(e['params']['k'] for e in baseline['edges'] if e['id'] == 'grocery_receive')
k_spoil  = next(e['params']['k'] for e in baseline['edges'] if e['id'] == 'pantry_spoil')
k_ptof   = next(e['params']['k'] for e in baseline['edges'] if e['id'] == 'pantry_to_fridge')

k_values  = np.array([0.0005, 0.001, 0.0015, 0.002, 0.003, 0.004])
pantry_final = []

for k in k_values:
    mod = set_edge_k(baseline, 'grocery_receive', float(k))
    d   = run_model(mod)
    pantry_final.append(d['pantry'].iloc[-1])

pantry_final = np.array(pantry_final)

# Analytical prediction (using steady-state Q_bank ≈ 26000)
pantry_theory = k_values * Q_star / (k_spoil + k_ptof)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: sensitivity curve
axes[0].plot(k_values * 1000, pantry_final,   'o-', label='GSSK (t=24 months)', lw=2)
axes[0].plot(k_values * 1000, pantry_theory,  '--', label='Analytical Q*', lw=1.5)
axes[0].set_xlabel('k_grocery (×10⁻³)')
axes[0].set_ylabel('pantry Q at t=24 (kg)')
axes[0].set_title('Pantry vs Grocery Spend Rate')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: time series for baseline vs doubled k
df_base   = run_model(baseline)
df_double = run_model(set_edge_k(baseline, 'grocery_receive', 2 * k_grocery_baseline))

axes[1].plot(df_base['time'],   df_base['pantry'],   label=f'k = {k_grocery_baseline} (baseline)')
axes[1].plot(df_double['time'], df_double['pantry'], label=f'k = {2*k_grocery_baseline} (doubled)')
axes[1].set_xlabel('Time (months)')
axes[1].set_ylabel('pantry Q (kg)')
axes[1].set_title('Pantry Trajectory: Baseline vs 2× k_grocery')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(REPO / 'docs' / 'examples' / 'household' / 'sensitivity.png', bbox_inches='tight')
plt.show()

# Verify doubling ratio
idx_base   = np.where(np.isclose(k_values, k_grocery_baseline, rtol=1e-9))[0][0]
idx_double = np.where(np.isclose(k_values, 2 * k_grocery_baseline, rtol=1e-9))[0][0]
ratio = pantry_final[idx_double] / pantry_final[idx_base]
print(f'k_grocery baseline: {k_grocery_baseline}')
print(f'pantry @ baseline : {pantry_final[idx_base]:.2f} kg')
print(f'pantry @ 2×k      : {pantry_final[idx_double]:.2f} kg')
print(f'ratio              : {ratio:.4f}  (expected ≈ 2.0)')

## 4. Information Carrier Dynamics

The information carrier is non-conserved — attention decays and decisions are forgotten.
Steady state for `household_attention`:

```
dQ_att/dt = k_replenish − (k_decay + k_att2dec × Q_news) × Q_att = 0
→ Q_att* = k_replenish / (k_decay + k_att2dec × Q_news)
         = 10 / (0.05 + 0.02 × 1) = 10 / 0.07 ≈ 142.9
```

In [ ]:
k_replenish = next(e['params']['k'] for e in baseline['edges'] if e['id'] == 'attention_replenish')
k_decay     = next(e['params']['k'] for e in baseline['edges'] if e['id'] == 'attention_decay')
k_att2dec   = next(e['params']['k'] for e in baseline['edges'] if e['id'] == 'attention_to_decisions')
Q_news      = next(n['value']       for n in baseline['nodes']  if n['id'] == 'news_inflow')

Q_att_star = k_replenish / (k_decay + k_att2dec * Q_news)
print(f'Analytical attention steady state: {Q_att_star:.2f}')
print(f'Simulated  attention at t=24     : {df["household_attention"].iloc[-1]:.2f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df['time'], df['household_attention'], label='household_attention')
axes[0].axhline(Q_att_star, ls='--', color='grey', label=f'Steady state ≈ {Q_att_star:.1f}')
axes[0].set_title('Attention Dynamics')
axes[0].set_xlabel('Time (months)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(df['time'], df['pending_decisions'], label='pending_decisions')
axes[1].set_title('Decision Backlog')
axes[1].set_xlabel('Time (months)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Summary Table

Final state at t = 24 months vs analytical steady-state predictions.

In [ ]:
records = [
    {'Node': 'bank_account',       'Carrier': 'money',       'Final (sim)': df['bank_account'].iloc[-1],
     'Steady state': Q_star,       'Unit': 'AUD'},
    {'Node': 'household_attention','Carrier': 'information', 'Final (sim)': df['household_attention'].iloc[-1],
     'Steady state': Q_att_star,   'Unit': 'decisions/mo'},
    {'Node': 'solar_battery',      'Carrier': 'energy',      'Final (sim)': df['solar_battery'].iloc[-1],
     'Steady state': float('nan'), 'Unit': 'kWh'},
    {'Node': 'pantry',             'Carrier': 'material',    'Final (sim)': df['pantry'].iloc[-1],
     'Steady state': float('nan'), 'Unit': 'kg'},
]

summary = pd.DataFrame(records).set_index('Node')
summary['Δ to SS (%)'] = ((summary['Final (sim)'] - summary['Steady state'])
                          / summary['Steady state'] * 100).round(2)
summary.style.format({'Final (sim)': '{:.2f}', 'Steady state': '{:.2f}',
                       'Δ to SS (%)': '{:+.2f}'})